In [1]:
from modules import creation_dataframe,CreationClipDataset,HeadClassifierClipModel,Train,CreationProcessedDataset, ClipEmbeddings
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor,CLIPImageProcessor,CLIPTokenizerFast,CLIPModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn.modules.loss import BCEWithLogitsLoss
import torch
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


In [2]:
#Creation of the dataframes
train_df=creation_dataframe("data/train.jsonl")

val_df=creation_dataframe("data/dev.jsonl")

In [3]:
#Creation of the first Clip Datasets to get the texts and images embeddings through the pretrained clip model
train_clip_dataset=CreationClipDataset(train_df)
val_clip_dataset=CreationClipDataset(val_df)

In [4]:
#Initialisation of Clip Processors
text_processor=CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
image_processor=CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
processor=CLIPProcessor(image_processor=image_processor, tokenizer=text_processor)

In [5]:
#Initialisation of the clip model, the device, and the batch size
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
batch_size=32


In [7]:
#Creation of the fisrt Clip Dataloaders to get the texts and images embeddings through the pretrained clip model

def collate_fn(batch):
    images=[b["images"] for b in batch]
    texts=[b["texts"] for b in batch]
    labels=[b["labels"] for b in batch]
    inputs=processor(images,texts,return_tensors="pt", padding=True, max_length=77, truncation=True, do_rescale=False)
    inputs["labels"]=torch.tensor(labels,dtype=torch.float32)
    return inputs

train_clip_dataloader=DataLoader(train_clip_dataset,batch_size=32,shuffle=True,collate_fn=collate_fn)
val_clip_dataloader=DataLoader(val_clip_dataset,batch_size=32,shuffle=True,collate_fn=collate_fn)

In [8]:
get_clip_embeddings=ClipEmbeddings(train_clip_dataloader,val_clip_dataloader,clip_model,device)
final_train_data,final_val_data=get_clip_embeddings()

In [9]:
train_dataset=CreationProcessedDataset(final_train_data)
val_dataset=CreationProcessedDataset(final_val_data)
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_dataloader=DataLoader(val_dataset,batch_size=32,shuffle=True)

In [21]:
model=HeadClassifierClipModel()

In [22]:
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight)
print(class_weight[1].item())

1.3934426229508197


In [26]:
n_epochs=10
batch_size=32
n_steps=(train_dataset.__len__()//batch_size)*n_epochs
optimizer=AdamW(model.parameters(),lr=1e-5,weight_decay=0.1)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=0.1*n_steps,num_training_steps=n_steps)
loss_fn=BCEWithLogitsLoss(pos_weight=class_weight[1])

In [24]:
training_object=Train(processor,model,loss_fn,optimizer,n_epochs,train_dataloader,val_dataloader,scheduler,device,batch_size, patience=10, min_improvement=0.05)

In [25]:
training_object.run_training()

2026-02-17 12:48:56.647 | INFO     | modules.Train:run_training:43 - Epoch 0 :
2026-02-17 12:49:01.189 | INFO     | modules.Train:run_training:100 - Epoch 0: Train Loss = 0.7675521936183586
2026-02-17 12:49:01.189 | INFO     | modules.Train:run_training:101 - Epoch 0: Train Accuracy = 0.6415294117647059
2026-02-17 12:49:01.189 | INFO     | modules.Train:run_training:102 - Epoch 0: Train F1 = 0.2941857771600649
2026-02-17 12:49:01.189 | INFO     | modules.Train:run_training:104 - Epoch 0: Validation Loss = 0.8470633998513222
2026-02-17 12:49:01.189 | INFO     | modules.Train:run_training:105 - Epoch 0: Validation Accuracy = 0.556
2026-02-17 12:49:01.189 | INFO     | modules.Train:run_training:106 - Epoch 0: Validation F1 = 0.3431952662721894
2026-02-17 12:49:01.189 | INFO     | modules.Train:run_training:43 - Epoch 1 :
2026-02-17 12:49:05.567 | INFO     | modules.Train:run_training:100 - Epoch 1: Train Loss = 0.6938123545028213
2026-02-17 12:49:05.567 | INFO     | modules.Train:run_trai